# Prueba de funcionamiento:
## Salud del sistema

In [1]:
import platform
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

print(f"1. Sistema: {platform.system()} | Arquitectura: {platform.machine()}")

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.binary_location = "/usr/bin/google-chrome"

try:
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("2. Driver y Chrome sincronizados.")
    driver.get("https://www.google.com")
    print(f"3.  Conexión a Internet: {driver.title}")
    driver.quit()
    print("4.  Entorno listo para trabajar.")
except Exception as e:
    print(f" ERROR EN EL RESET: {e}")

1. Sistema: Linux | Arquitectura: x86_64
2. Driver y Chrome sincronizados.
3.  Conexión a Internet: Google
4.  Entorno listo para trabajar.


# Scraper Dinámico

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("https://www.chileautos.cl/vehiculos/autos-veh%C3%ADculo/usado-tipo/")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, ".card-body"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, ".card-body")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            dato_identificador = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("sv-title")
            dato_valor = bloque.find_element(By.CLASS_NAME, ".item-price").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "metadata": "Capturado via Selenium"
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Nodo (Paginación) ---
        try:
            # Buscamos el activador del siguiente nodo de datos
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()
#Cambios en los datos de salida
for dato in datos_finales: #Recorremos la estructura con los datos almacenados en el computador
    dato['valor'] = float(dato['valor'].replace('£', '')) #Quito el símbolo de moneda que se obtiene y se convierte en float
    print(dato.get("valor")) #Obtengo el valor de ese dato modificado que datos tengo q cambiar en base a esta pagina

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

